# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [2]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('/content/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [3]:
# Task 1
# Fossil fuels only -> reused in Task 3
df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

# CVD-safe colours (Okabe-Ito): vermillion / sky blue / yellow differ in hue AND lightness
fossil_colors = {'Coal': '#D55E00', 'Oil': '#56B4E9', 'Natural Gas': '#F0E442'}
PARENT_GREY = '#DDDDDD'

# Numbers for the insight title
region_tot  = df_fossil.groupby('Region')['TWh'].sum().sort_values(ascending=False)
country_tot = df_fossil.groupby('Country')['TWh'].sum().sort_values(ascending=False)
top_region, second_region = region_tot.index[0], region_tot.index[1]
top_country = country_tot.index[0]
ratio = region_tot[top_region] / region_tot[second_region]

# Step 1: Plotly Express base chart
fig = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',                                  # additive -> parents sum correctly
    color='Source',
    color_discrete_map=fossil_colors,
    labels={'Source': 'Fossil source', 'TWh': 'Energy (TWh)'},
    height=700, width=1100,
    template='plotly_white',
)

# Step 2: Customisation
fig.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',  # TWh only, no percentages
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
    textfont=dict(color='#222222'),
    marker_line_color='white', marker_line_width=1,
)

# Grey out parent nodes: leaves sit three levels deep (Region/Country/Source), everything above is a parent
t = fig.data[0]
t.marker.colors = [fossil_colors[lab] if i.count('/') == 2 else PARENT_GREY
                   for i, lab in zip(t.ids, t.labels)]

fig.update_layout(
    title=dict(
        text=(f"<b>{top_region} is the most fossil-dependent region</b>: {region_tot[top_region]:,.0f} TWh, "
              f"{ratio:.1f}x {second_region}<br>"
              f"<span style='font-size:13px'>{top_country} is the largest single country "
              f"at {country_tot[top_country]:,.0f} TWh</span>"),
        x=0, xanchor='left'),
    font=dict(family='Arial', size=12),
    margin=dict(t=90, b=50),
    annotations=[dict(
        text=("Area = TWh.  Colour = fossil source: "
              "<span style='color:#D55E00'>&#9632;</span> Coal  "
              "<span style='color:#56B4E9'>&#9632;</span> Oil  "
              "<span style='color:#F0E442'>&#9632;</span> Natural Gas  "
              "|  Grey = region / country totals"),
        xref='paper', yref='paper', x=0, y=-0.05, xanchor='left',
        showarrow=False, font=dict(size=11, color='grey'))],
)
fig.show()

## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [4]:
# Task 2 - charts
# Data prep: low-carbon (Nuclear + Hydro) TWh per country
df_lc = (df.loc[df['Source_Type'] == 'Low-carbon']
           .groupby('Country', as_index=False)['TWh'].sum()
           .sort_values('TWh', ascending=False))
df_lc['All'] = 'Low-carbon'                       # dummy root node

BLUE, GREY = '#2E75B6', '#DDDDDD'                 # course palette (CVD-safe)
c1, c2, c3 = df_lc['Country'].iloc[:3]
t1, t2, t3 = df_lc['TWh'].iloc[:3]
top4, total = df_lc['TWh'].iloc[:4].sum(), df_lc['TWh'].sum()
top4_names = ', '.join(df_lc['Country'].iloc[:3]) + ' and ' + df_lc['Country'].iloc[3]

# ---- Chart A: treemap ----
fig = px.treemap(
    df_lc, path=['All', 'Country'], values='TWh',
    height=600, width=1100, template='plotly_white',
)
fig.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
    textfont=dict(color='white'),
    marker_line_color='white', marker_line_width=1,
)
t = fig.data[0]
t.marker.colors = [GREY if p == '' else BLUE for p in t.parents]   # root grey, countries blue
fig.update_layout(
    title=dict(text=(f"{top4_names} generate {top4:,.0f} of the {total:,.0f} TWh "
                     f"of low-carbon energy"), x=0, xanchor='left'),
    font=dict(family='Arial', size=12), margin=dict(t=70),
)
fig.show()

# ---- Chart B: horizontal bar ----
bar = df_lc.sort_values('TWh', ascending=True)    # ascending: largest bar ends up at the top

fig = px.bar(
    bar, x='TWh', y='Country', orientation='h',
    text='TWh', color_discrete_sequence=[BLUE],
    labels={'TWh': 'Low-carbon energy (TWh)', 'Country': ''},
    height=550, width=1000, template='plotly_white',
)
fig.update_traces(texttemplate='%{x:,.0f} TWh', textposition='outside', cliponaxis=False)
fig.update_layout(
    title=dict(text=f"{c1} leads low-carbon energy at {t1:,.0f} TWh, ahead of {c2} ({t2:,.0f}) and {c3} ({t3:,.0f})",
               x=0, xanchor='left'),
    font=dict(family='Arial', size=12),
    xaxis=dict(range=[0, t1 * 1.15], showgrid=False),
    yaxis=dict(title=''), bargap=0.3, margin=dict(t=70),
)
fig.show()

### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

The **bar chart**. All bars share one baseline and are sorted, so the reader compares *lengths*, which the eye judges accurately, and the order top to bottom is the ranking. In the treemap the reader has to compare *areas* of rectangles with different shapes, which is much harder: France vs Norway vs Canada look similar, and the middle of the ranking (Mexico, Germany, UK, Japan, China) is nearly impossible to order by eye. The tiny tiles (e.g. Australia at 448 TWh) are also hard to read.

The treemap's strengths are magnitude within a hierarchy and the overall impression that a few countries dominate. Here there is only **one level** (country) and no nested structure, so the treemap has nothing extra to show, and the bar chart is the better choice for a ranking.

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [5]:
# Task 3
# Aggregate fossil TWh by Region x Source (reusing df_fossil from Task 1), including zero rows
reg = (df_fossil.groupby(['Region', 'Source'])['TWh'].sum()
                .unstack(fill_value=0)                      # Middle East has no coal -> 0
                .reset_index()
                .melt(id_vars='Region', var_name='Source', value_name='TWh'))
reg['Share'] = reg['TWh'] / reg.groupby('Region')['TWh'].transform('sum')   # shares of each region's fossil total

# Sort regions by coal share. Ascending list: Plotly draws the first category at the bottom,
# so the highest coal share ends up at the top.
coal_share = reg.loc[reg['Source'] == 'Coal'].set_index('Region')['Share'].sort_values()
region_order = coal_share.index.tolist()
coal_region, coal_top = coal_share.index[-1], coal_share.iloc[-1]

# Labels on the coal segments only (skip 0%)
reg['Label'] = np.where((reg['Source'] == 'Coal') & (reg['Share'] > 0),
                        reg['Share'].map('{:.0%}'.format), '')

# One accent: coal keeps its Task 1 colour; oil and natural gas become greys
accent_mapping = {'Coal': fossil_colors['Coal'], 'Oil': '#BBBBBB', 'Natural Gas': '#DDDDDD'}

fig = px.bar(
    reg, x='Share', y='Region', color='Source', text='Label',
    orientation='h',
    color_discrete_map=accent_mapping,
    category_orders={'Region': region_order,
                     'Source': ['Coal', 'Oil', 'Natural Gas']},   # coal on the baseline
    labels={'Share': 'Share of fossil energy', 'Source': 'Fossil source'},
    hover_name='Region', hover_data={'Label': False, 'Share': ':.1%', 'TWh': ':,.0f'},
    height=500, width=1000, template='plotly_white',
)
fig.update_traces(
    textposition='inside', insidetextanchor='start',
    textfont=dict(color='white', size=12),
    marker_line_color='white', marker_line_width=1,
)
fig.update_layout(
    title=dict(text=f"{coal_region} is the most coal-dependent region: {coal_top:.0%} of its fossil energy is coal",
               x=0, xanchor='left'),
    font=dict(family='Arial', size=12),
    bargap=0.25,
    legend=dict(orientation='h', x=0, y=1.08),
    xaxis=dict(visible=False, range=[0, 1]),
    yaxis=dict(title=''),
    margin=dict(t=100),
)
fig.add_annotation(x=1, y=0, xref='x', yref='paper', xanchor='right', yanchor='top', yshift=-4,
                   text='100% = each region\'s total fossil energy (coal + oil + natural gas)',
                   showarrow=False, font=dict(size=11, color='grey'))
fig.show()

### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

**Treemap:** *Where is the most fossil energy burned in absolute terms?* Area is TWh, so Asia (about 35,000 TWh) visibly dwarfs Africa (about 8,600 TWh), and you can see which countries inside each region drive the total. The stacked bar cannot show this, because every bar is rescaled to 100% and the regions all look the same size.

**100% stacked bar:** *Which region depends most on coal relative to its own fossil mix?* Africa is about 78% coal versus about 49% for Asia and about 6% for Europe, ranked on a common baseline. In the treemap the coal tiles are sized by TWh, so a small region with a very coal-heavy mix (Africa) looks unimportant, and the shares cannot be compared across regions of different size.